# Structural neurotransmitter fingerprinting

In this tutorial, you will explore the structural neurotransmitter fingerprinting (SNTF) functionalities of Lacuna using the CLI.

**What you'll learn**:

- Fetch the neurotransmitter PET atlas and a structural connectome
- Prepare the atlas and precompute endpoint weights
- Compute NT-weighted structural disconnectivity scores
- Filter by neurotransmitter system

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-petersen/lacuna/blob/main/docs/tutorials/07-structural-neurotransmitter-fingerprinting.ipynb)

## Colab

Note: Colab provides limited computational resources. While these tutorials are designed to operate within those constraints, some Lacuna functionality cannot be fully demonstrated in this environment and requires access to higher-performance computing infrastructure.

Ignore this if you run this notebook locally.

In [ ]:
# --- Conda setup for Google Colab ---
# Lacuna's structural neurotransmitter fingerprinting relies on MRtrix3.
# However, Colab does not provide MRtrix3 out of the box.
# The condacolab package provides a workaround.
# This cell installs condacolab, which will restart the kernel.
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# Run this after the kernel restart. 
import sys

if 'google.colab' in sys.modules:

    import condacolab
    import subprocess
    condacolab.check()
    subprocess.run(["conda", "install", "-y", "-c", "mrtrix3", "mrtrix3"], check=True)
else:
    print("Not running in Colab — skipping condacolab setup.")

## Setup

In [ ]:
# Install Lacuna from GitHub
!pip install git+https://github.com/m-petersen/lacuna

# Install MRtrix3 via conda
!conda install -y -c mrtrix3 mrtrix3

Get the tutorial data.

In [ ]:
# Get tutorial data
!lacuna tutorial /tmp/tutorial_bids --force

Check whether MRtrix3 is properly installed as it is required for the analysis.

In [ ]:
!mrinfo --version

## Fetch data

SNTF requires two data sources:

1. **Neurotransmitter PET atlas** — PET receptor/transporter density maps from normative cohorts (from [OSF](https://osf.io/yz9mb/))
2. **Structural connectome** — A normative tractogram of white matter fiber bundles (e.g., HCP1065)

In [ ]:
# Fetch NT atlas
!lacuna fetch ntatlas \
    --output-dir /tmp/ntatlas_data

In [ ]:
# Fetch structural connectome
!lacuna fetch hcp1065 \
    --output-dir /tmp/hcp1065_data

## Prepare the atlas and endpoint weights

Before running SNTF, two preparation steps are needed:

1. **Prepare NT atlas** — Average per target, z-score, and cache
2. **Precompute endpoint weights** — Sample NT atlas values at all streamline endpoints and cache the result

Step 2 is the most computationally expensive part but only needs to be done once per connectome.

In [ ]:
# Step 1: Prepare the NT atlas
!lacuna prepare lntf \
    --source-dir /tmp/ntatlas_data \
    --cache-dir /tmp/ntatlas_cache

In [ ]:
# Step 2: Precompute endpoint weights for the tractogram
!lacuna prepare sntf \
    --atlas-cache-dir /tmp/ntatlas_cache \
    --connectome-path /tmp/hcp1065_data/hcp1065_1mm.tck \
    --cache-dir /tmp/sntf_cache

## Analysis

Structural neurotransmitter fingerprinting combines structural disconnection with neurotransmitter information. It:

1. Filters the normative tractogram by the lesion mask to identify disconnected streamlines
2. Looks up NT atlas values at the endpoints of those disconnected streamlines
3. Scores each neurotransmitter target based on the endpoint values

This answers: **what NT-weighted structural connectivity does the lesion disrupt?**

A high score for a given target indicates that the lesion disconnects pathways whose endpoints are rich in that neurotransmitter.

Run the analysis.

In [ ]:
!lacuna run sntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_sntf/ \
    --connectome-path /tmp/hcp1065_data/hcp1065_1mm.tck \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_cache

List the outputs.

In [ ]:
!ls /tmp/outputs_sntf/sub-01/ses-01/anat/

## Filter by neurotransmitter system

You can restrict the analysis to specific neurotransmitter systems using the `--targets` flag:

| Preset | Targets |
|--------|--------|
| `dopaminergic` | D1, D23, DAT, FDOPA |
| `serotonergic` | 5HT1a, 5HT1b, 5HT2a, 5HT4, 5HT6, 5HTT |
| `cholinergic` | VAChT, M1, A4B2 |
| `monoaminergic` | D1, D23, DAT, 5HT1a, 5HT1b, 5HT2a, 5HT4, 5HT6, 5HTT, NET |
| `all` | All available targets (default) |

In [ ]:
!lacuna run sntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_sntf_serotonin/ \
    --connectome-path /tmp/hcp1065_data/hcp1065_1mm.tck \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_cache \
    --targets serotonergic

## Run on multiple subjects

Lacuna supports processing multiple subjects within a single run. SNTF requires MRtrix3 for tractogram filtering, so processing speed depends on the tractogram size.

Using precomputed endpoint weights (`lacuna prepare sntf`) significantly speeds up batch processing since the NT sampling step is cached.

In [ ]:
!lacuna run sntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_sntf_all/ \
    --connectome-path /tmp/hcp1065_data/hcp1065_1mm.tck \
    --mask-space MNI152NLin6Asym \
    --atlas-cache-dir /tmp/ntatlas_cache

In [ ]:
!ls /tmp/outputs_sntf_all/